# Ascend 950 硬件架构与执行域

本节打开 Device 的"引擎盖"，建立后续所有编程内容都依赖的硬件地图：AI Core 的组成、AIC 与 AIV 两类执行域、存储层级与数据路径、以及搬运与计算的异步关系。读完本节，你看到的将不再是一块"NPU"，而是一张可以对着编程概念逐一对照的硬件结构图。


## 1. AI Core：从三个视角理解

Ascend 950 的计算发生在 **AI Core** 上。理解一个 AI Core，可以从三个视角入手——**计算单元**（谁在算）、**存储单元**（数据放在哪）、**搬运单元**（数据怎么流动）。

<img style="display:block; margin-left:0; margin-right:auto;" src="./images/hardware_architecture_950.svg" alt="AI Core 硬件架构" width="700px">

（图片来源：PyPTO 官方文档《抽象硬件架构》，图中黑色实线为主要数据流，橙色虚线为指令流）

一个 NPU 芯片上有多个 AI Core，可以并行处理不同的数据分片；AI Core 内部则包含 Scalar、Vector、Cube、片上存储和数据搬运等单元。


## 2. 计算单元：Scalar、Vector 与 Cube

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">单元</th><th style="text-align:left;">主要作用</th><th style="text-align:left;">PyPTO Pro 中的对应表达</th></tr>
<tr><td><b>Scalar 单元</b></td><td>地址计算、循环控制等标量逻辑，向计算和搬运流水发射指令</td><td>Kernel 中的 Python 控制逻辑</td></tr>
<tr><td><b>Vector 单元（AIV 上）</b></td><td>矢量运算</td><td>Tile 计算、Reg 计算</td></tr>
<tr><td><b>Cube 单元（AIC 上）</b></td><td>矩阵乘加等矩阵运算</td><td>Cube 计算</td></tr>
</table>

三种计算方式的数据载体各不相同：

- **Tile 计算**：以 UB（Unified Buffer）中的 Tile 为计算数据载体；
- **Reg 计算**：将 UB 中的数据加载到 Vector Register File，中间结果可以保留在寄存器中；
- **Cube 计算**：从 L0A Buffer 和 L0B Buffer 读取矩阵操作数，将累加结果写入 L0C Buffer。


## 3. 执行域：AIC 与 AIV

AI Core 分为两类执行域（Execution Domain），这是理解 Pro 多核编程的关键概念：

- **AIC（AI Cube）执行域**：主要执行 Cube 侧的 SIMD 矩阵计算；
- **AIV（AI Vector）执行域**：主要执行 Vector 侧的 SIMD 或 SIMT 计算。

一个 Kernel 可以只使用 Vector 资源（如逐元素算子），只使用 Cube 资源（如纯矩阵乘），也可以让 AIC 与 AIV 协同工作（如"矩阵乘 + 激活"的融合算子）。多核编程时，Host 侧通过 `block_dim` 配置核数，各核以 SPMD 方式并行处理不同数据分片——这些内容将在第 2 章和第 4 章展开。


## 4. 存储层级与数据路径

GM（Global Memory）位于 AI Core 之外，用于保存 Kernel 的输入、输出及 Workspace。AIV 和 AIC 各有片上存储，计算前需要将数据搬入相应的存储层级：

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">执行侧</th><th style="text-align:left;">主要片上存储</th><th style="text-align:left;">典型数据路径</th></tr>
<tr><td>AIV Tile 计算</td><td>UB</td><td>GM → UB → Vector 单元 → UB → GM</td></tr>
<tr><td>AIV Reg 计算</td><td>UB、Vector Register File</td><td>GM → UB → Vector Register File → UB → GM</td></tr>
<tr><td>AIC Cube 计算</td><td>L1 Buffer、L0A/L0B/L0C Buffer</td><td>GM → L1 → L0A/L0B → Cube 单元 → L0C → GM</td></tr>
</table>

> 表中的路径表示一次计算所涉及的**主要**存储层级，不代表所有搬运和计算必须串行完成。离计算单元越近的存储，容量越小、带宽越高——"把数据分级摆放"正是高性能算子设计的核心手段，也是 Tile 与片上内存规划（第 2 章）要解决的问题。


## 5. 搬运单元、异步流水与同步

**MTE 等搬运单元**负责 GM 与片上 Buffer、以及不同片上 Buffer 之间的数据流转：AIV 的 GM 与 UB 搬运主要使用 MTE2、MTE3；Reg 计算还使用 DMA 单元在 UB 与 Vector Register File 之间搬运数据；AIC 的矩阵数据通常经 MTE2 进入 L1 Buffer，再经 MTE1 进入 L0A/L0B，结果由 FIX 搬出。

**指令流与数据流是两回事**：Scalar 单元把搬运和计算任务**发射**到相应流水后，各流水按自己的节奏执行——源码中的先后顺序，不等同于硬件上的完成顺序。这带来一个直接推论：

- 当一条流水生产的数据要由另一条流水消费时，必须建立**同步依赖**。例如：Vector 计算读取 UB 数据之前，相关搬入必须完成；计算结果搬回 GM 之前，相关计算也必须完成；
- 同步信号只约束执行顺序，不承担数据搬运。

PyPTO Pro 提供了 TileGroup 与自动 Mutex（`auto_mutex`）管理这类依赖，也提供显式同步接口——这是第 2 章"同步机制"小节的主题。此处你只需记住一句话：**搬运和计算是异步并行的，谁消费数据，谁就要等谁生产完**。


### 把第一段代码放回硬件图

对 [8,256] 的加法，三个 Tile 位于当前 Vector 核的 UB：MTE2 搬入 x/y，Vector 做加法，MTE3 写出结果。不同 Vector 核可以使用相同 UB 偏移，因为它们各有自己的存储；如果所有核仍向同一个 GM 输出起点写入，则会发生输出重叠。

再看矩阵乘：A/B 先到 L1，再分别到 L0A/L0B，由 Cube 相乘并写入 L0C。两个问题要分开回答：“元素放在哪个 Buffer”决定操作数是否可达；“谁先完成”决定消费者是否能安全读取。同步不能把数据从 L0C 变到 UB，搬运也不能代替跨流水的完成关系。

**自查**：逐元素加法的输入不能直接作为 Cube 的矩阵操作数；这既涉及计算语义，也涉及数据路径与布局。


## 6. 课后练习

1. （选择）矩阵乘的输入操作数由 Cube 单元读取，它们存放在：
   A. UB<br/>B. L0A Buffer 与 L0B Buffer<br/>C. Vector Register File<br/>D. GM（直接读取）
2. （填空）AIV 执行域承担 Vector 侧计算，其 GM 与 UB 之间的搬运主要使用 MTE______（搬入）与 MTE______（搬出）。
3. （判断）源码中先写"数据搬入"、后写"计算"，硬件上就一定先完成搬入再开始计算。（对/错，并说明理由）

In [ ]:
!cat ./answer/01.03_answer.txt

## 7. 本节小结

- AI Core = **计算单元（Scalar/Vector/Cube）+ 片上存储（UB/RegFile/L1/L0）+ 搬运单元（MTE/DMA/FIX）**；GM 位于 AI Core 之外；
- 两类执行域：**AIC 管矩阵，AIV 管矢量**；多核 SPMD 并行由 `block_dim` 配置；
- 数据路径遵循"**GM → 片上逐级搬近计算单元 → 计算 → 逐级搬回**"的分层结构；
- **搬运与计算异步**：消费数据前必须同步——这句话将贯穿整门课程。

下一节介绍 PyPTO Pro：一个让你用 Python 语法驾驭这套硬件的编程范式。